In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors


In [ ]:
# 2. Create TensorDataset objects




In [ ]:
# 3. Create DataLoaders




In [ ]:
# 4. Print shape of one batch



In [ ]:
# 5. Display sample images



In [ ]:
import torch
import numpy as np

# --- Convert to tensors (safe if already tensors) ---
X_train = torch.as_tensor(X_train)
y_train = torch.as_tensor(y_train)
X_test  = torch.as_tensor(X_test)
y_test  = torch.as_tensor(y_test)

# --- Fix common label shapes ---
# Case: labels are shape (N,1) -> make (N,)
if y_train.dim() > 1 and y_train.shape[-1] == 1:
    y_train = y_train.squeeze(-1)
if y_test.dim() > 1 and y_test.shape[-1] == 1:
    y_test = y_test.squeeze(-1)

# Case: labels are one-hot (N, num_classes) -> convert to class indices
if y_train.dim() == 2 and y_train.size(1) > 1:
    y_train = y_train.argmax(dim=1)
if y_test.dim() == 2 and y_test.size(1) > 1:
    y_test = y_test.argmax(dim=1)

# --- Combine labels to analyze full dataset classes ---
y_all = torch.cat([y_train, y_test], dim=0)

# 1) Number of unique labels
unique_labels = torch.unique(y_all)
num_unique_labels = unique_labels.numel()

# 2) Shape of input data
input_shape_train = tuple(X_train.shape)
input_shape_test  = tuple(X_test.shape)

# Per-sample shape (excluding N)
per_sample_shape = tuple(X_train.shape[1:])

# 3) Max / 4) Min across the FULL dataset (train+test)
X_all = torch.cat([X_train.reshape(X_train.size(0), -1),
                   X_test.reshape(X_test.size(0), -1)], dim=0)
max_val = X_all.max().item()
min_val = X_all.min().item()

print(" Task 1 Results")
print("-" * 60)
print(f"1) Unique labels count: {num_unique_labels}")
print(f"   Unique labels: {unique_labels.tolist()}")
print(f"2) Train input shape: {input_shape_train}")
print(f"   Test  input shape: {input_shape_test}")
print(f"   Per-sample shape:  {per_sample_shape}")
print(f"3) Max value in dataset: {max_val}")
print(f"4) Min value in dataset: {min_val}")


In [ ]:
# Task 1: Write your model class here:



In [ ]:
# Task 2: Write your training loop here:

In [ ]:
# Task 3: Write your validation loop here:

In [ ]:
# Task 4: Define device, model, loss, optimizer:

In [ ]:
# Task 5: Start training for 20 epochs:

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# Convert types for NN training
X_train = X_train.float()
X_test  = X_test.float()
y_train = y_train.float()
y_test  = y_test.float()

# If input is (N, H, W), add channel -> (N, 1, H, W)
if X_train.dim() == 3:
    X_train = X_train.unsqueeze(1)
    X_test  = X_test.unsqueeze(1)

# --- Normalize using TRAIN stats only (important!) ---
train_mean = X_train.mean()
train_std  = X_train.std() + 1e-8

X_train = (X_train - train_mean) / train_std
X_test  = (X_test  - train_mean) / train_std

# Create validation split from training set (so test remains "final exam evaluation")

batch_size = 64

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_train, X_train), batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

print("Loaders ready")
print("Train batches:", len(train_loader))
print("Val batches:  ", len(val_loader))
print("Test batches: ", len(test_loader))
print("Example batch X shape:", next(iter(train_loader))[0].shape)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch

num_classes = int(torch.unique(torch.cat([y_train, y_test])).numel())
in_shape = tuple(X_train.shape[1:])  # use train split after channel fix

class CompactNet(nn.Module):
    """
    Max 5 LEARNABLE layers.
    CNN version (4 learnable layers):
      1) Conv2d
      2) Conv2d
      3) Linear
      4) Linear
    """
    def __init__(self, in_shape, num_classes):
        super().__init__()
        C, H, W = in_shape

        self.conv1 = nn.Conv2d(C, 16, kernel_size=3, padding=1)  # (1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # (2)
        self.pool  = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(32, 64)          # (3)
        self.fc2 = nn.Linear(64, num_classes) # (4)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CompactNet(in_shape, num_classes).to(device)

print(model)
print(" Device:", device)


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# Handle imbalance with class weights (from y_tr only)
class_counts = torch.bincount(y_train)
class_weights = (class_counts.sum() / (class_counts + 1e-8))
class_weights = class_weights / class_weights.sum()
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def eval_loader(model, loader):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    all_preds, all_true = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * xb.size(0)
            total += xb.size(0)

            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()

            all_preds.append(preds.cpu())
            all_true.append(yb.cpu())

    avg_loss = total_loss / total
    acc = correct / total

    all_preds = torch.cat(all_preds).numpy()
    all_true  = torch.cat(all_true).numpy()
    macro_f1 = f1_score(all_true, all_preds, average="macro")

    return avg_loss, acc, macro_f1


epochs = 15
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    total_train = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        total_train += xb.size(0)

    train_loss = running_loss / total_train
    val_loss, val_acc, val_f1 = eval_loader(model, val_loader)

    print(f"Epoch {epoch:02d}/{epochs} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"Val Macro-F1: {val_f1:.4f}")

# Final test evaluation
test_loss, test_acc, test_f1 = eval_loader(model, test_loader)
print("\n FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.4f}")
print(f"Test Macro-F1: {test_f1:.4f}")


In [ ]:
# Task 1: Write your code here:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: